# TravelTide – Data Exploration

## 1. Project Objective

This notebook is dedicated to the initial exploration and understanding of the TravelTide dataset.

### Objectives

- Load and inspect the source data.
- Understand the dataset structure and data types.
- Identify missing values and potential data quality issues.
- Explore relevant variables and their distributions.
- Prepare the data for subsequent cleaning and analysis.

### Data Management Principles

- Original source data must remain unchanged.
- All data cleaning and transformations must be performed on working copies.
- Intermediate and final datasets must be stored separately.

## 2. Environment Setup

This section prepares the Python environment for the TravelTide data exploration.

### 2.1 Import Libraries

Import the required Python libraries for data processing, numerical calculations, and data visualization.

In [86]:
# 2.1 Import libraries for data exploration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Configure matplotlib
plt.style.use("default")

print("TravelTide data exploration environment is ready.")


TravelTide data exploration environment is ready.


## 3. Project Structure and Paths

This section defines the project directory structure and establishes reusable paths for data access.

### 3.1 Define Project Paths

Define the project root and all relevant data directories using pathlib.

All paths are derived from the project root to ensure consistent and portable file access.

In [87]:
# 3.1 Import Path for cross-platform file handling
from pathlib import Path

# Define project root directory
PROJECT_ROOT = Path.cwd().parent

# Define data directories
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
ORIGINAL_DIR = INPUT_DIR / "original"
VERSIONS_DIR = INPUT_DIR / "versions"

WORKING_DIR = DATA_DIR / "working"
CLEANING_DIR = WORKING_DIR / "cleaning"
TRANSFORMATIONS_DIR = WORKING_DIR / "transformations"
ANALYSIS_DIR = WORKING_DIR / "analysis"

OUTPUT_DIR = DATA_DIR / "output"
REPORTS_DIR = OUTPUT_DIR / "reports"
FINAL_DATASETS_DIR = OUTPUT_DIR / "final_datasets"

# Display project root
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide
Data directory: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide\data


### 3.2 Verify Project Directories

Verify that all required project data directories exist before starting the data exploration.

In [88]:
# 3.2 Verify project data directories

directories = {
    "Original data": ORIGINAL_DIR,
    "Data versions": VERSIONS_DIR,
    "Data cleaning": CLEANING_DIR,
    "Data transformations": TRANSFORMATIONS_DIR,
    "Data analysis": ANALYSIS_DIR,
    "Reports": REPORTS_DIR,
    "Final datasets": FINAL_DATASETS_DIR,
}

for name, path in directories.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name}: {status}")

Original data: OK
Data versions: OK
Data cleaning: OK
Data transformations: OK
Data analysis: OK
Reports: OK
Final datasets: OK


### 3.3 Display Environment Information

Display the Python version, installed library versions, active Python interpreter, and project root directory.

In [89]:
# 3.3 Display project environment information

import sys
import pandas as pd
import numpy as np
import matplotlib

print("TravelTide Environment Information")
print("-----------------------------------")
print(f"Python version: {sys.version.split()[0]}")
print(f"Python executable: {sys.executable}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Project root: {PROJECT_ROOT}")

TravelTide Environment Information
-----------------------------------
Python version: 3.14.6
Python executable: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide\.venv\Scripts\python.exe
NumPy version: 2.5.4
Pandas version: 3.1.0
Matplotlib version: 3.11.2
Project root: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide


## 4. Database Connection and Local Data Inventory

This section establishes the connection to the central Neon PostgreSQL database and manages local copies of the original source data.

The objective is to ensure that all required database tables are available locally while protecting existing original copies from accidental overwriting.

### Data Management Principles

- Neon is the central source of truth.
- Original database tables must never be modified by the analysis process.
- Existing local original copies must never be overwritten automatically.
- Only tables without an existing local copy are downloaded.
- Local copies of tables no longer available in Neon are retained.
- All imported data must be documented and traceable.
- Data from different import dates must be identified to prevent unintended mixed-version analyses.

### Import and Inventory Process

1. Establish a connection to the Neon database.
2. Retrieve the available database tables.
3. Scan the local original data directory.
4. Compare database tables with existing local copies.
5. Display the current database and local data status.
6. Import missing local copies without overwriting existing files.
7. Display an import summary, including skipped tables, newly imported tables, and errors.

### Section Overview

- **4.1 Connect to Neon Database**
- **4.2 Retrieve Database Table Information**
- **4.3 Scan Local Original Copies**
- **4.4 Compare Database and Local Tables**
- **4.5 Display Database and Local Data Status**
- **4.6 Import Missing Local Copies**
- **4.7 Display Import Summary**

# 4.0 Test .env Configuration

In [90]:
# Test .env Configuration

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Required variables
required_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

print("TRAVELTIDE – ENVIRONMENT CONFIGURATION TEST")
print("=" * 50)

for variable in required_variables:
    value = os.getenv(variable)

    status = "OK" if value else "MISSING"

    print(f"{variable}: {status}")

print("=" * 50)

TRAVELTIDE – ENVIRONMENT CONFIGURATION TEST
DB_HOST: OK
DB_PORT: OK
DB_NAME: OK
DB_USER: OK
DB_PASSWORD: OK


### 4.1 Connect to Neon Database

Establish a secure connection to the central Neon PostgreSQL database using the connection parameters stored in the local environment configuration file.

The connection must be validated before any database operations are performed.

In [91]:
# 4.1 Connect to Neon Database

# Import required libraries
import os
import psycopg
from dotenv import load_dotenv

# Reset database connection status
database_connected = False
database_tables = []
database_info_available = False

# Load environment variables
load_dotenv()

# Define required database configuration variables
required_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

# Check database configuration
missing_variables = [
    variable for variable in required_variables
    if not os.getenv(variable)
]

print("TRAVELTIDE – DATABASE CONNECTION")
print("=" * 45)

if missing_variables:
    print("Database configuration: INCOMPLETE")
    print("Missing variables:", ", ".join(missing_variables))
    print("\nConnection test: SKIPPED")
    print("Please configure the missing values in .env.")

else:
    print("Database configuration: COMPLETE")

    # Build database connection parameters
    connection_parameters = {
        "host": os.getenv("DB_HOST"),
        "port": os.getenv("DB_PORT"),
        "dbname": os.getenv("DB_NAME"),
        "user": os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
        "connect_timeout": 10
    }

    # Test database connection
    try:
        with psycopg.connect(**connection_parameters) as connection:
            with connection.cursor() as cursor:
                cursor.execute("SELECT version();")
                database_version = cursor.fetchone()[0]

        database_connected = True

        print("Database connection: SUCCESSFUL")
        print("PostgreSQL version:", database_version)

    except psycopg.Error as error:
        print("Database connection: FAILED")
        print("Error:", error)

print("=" * 45)

TRAVELTIDE – DATABASE CONNECTION
Database configuration: COMPLETE
Database connection: SUCCESSFUL
PostgreSQL version: PostgreSQL 15.19 (f55ec65) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit


### 4.2 Retrieve Database Table Information

Retrieve metadata about all available tables in the Neon PostgreSQL database.

The database inventory identifies the available schemas, table names, and estimated row counts.

This information provides the foundation for comparing database tables with existing local original copies.

In [92]:
# 4.2 Retrieve Database Table Information

print("TRAVELTIDE – DATABASE TABLE INVENTORY")
print("=" * 45)

if not database_connected:
    print("Database inventory: SKIPPED")
    print("Reason: No active database connection.")

else:
    try:
        with psycopg.connect(**connection_parameters) as connection:
            with connection.cursor() as cursor:
                cursor.execute("""
                    SELECT
                        table_schema,
                        table_name
                    FROM information_schema.tables
                    WHERE table_type = 'BASE TABLE'
                      AND table_schema NOT IN (
                          'pg_catalog',
                          'information_schema'
                      )
                    ORDER BY table_schema, table_name;
                """)

                database_tables = cursor.fetchall()

        database_info_available = True

        print(f"Tables found: {len(database_tables)}")

        for schema, table in database_tables:
            print(f"{schema}.{table}")

    except psycopg.Error as error:
        database_tables = []
        database_info_available = False

        print("Database inventory: FAILED")
        print("Error:", error)

print("=" * 45)

TRAVELTIDE – DATABASE TABLE INVENTORY
Tables found: 4
public.flights
public.hotels
public.sessions
public.users


### 4.3 Scan Local Original Copies

Scan the local original data directory and identify all available database table copies.

The inventory records file names, file formats, file sizes, and modification timestamps.

Existing original copies must remain unchanged and must never be overwritten automatically.

In [93]:
# 4.3 Scan Local Original Copies

print("TRAVELTIDE – LOCAL ORIGINAL DATA INVENTORY")
print("=" * 50)

# Identify existing CSV files in the original data directory
original_files = sorted(ORIGINAL_DIR.glob("*.csv"))

# Create local data inventory
local_inventory_data = [
    {
        "File Name": file.name,
        "File Format": file.suffix.lower(),
        "File Size (KB)": round(file.stat().st_size / 1024, 2),
        "Last Modified": pd.to_datetime(
            file.stat().st_mtime, unit="s"
        )
    }
    for file in original_files
]

# Convert inventory into a DataFrame
local_inventory = pd.DataFrame(
    local_inventory_data,
    columns=[
        "File Name",
        "File Format",
        "File Size (KB)",
        "Last Modified"
    ]
)

# Display inventory
print(f"Local original copies found: {len(local_inventory)}")
display(local_inventory)

print("=" * 50)

TRAVELTIDE – LOCAL ORIGINAL DATA INVENTORY
Local original copies found: 4


,File Name,File Format,File Size (KB),Last Modified
0,public__flights.csv,.csv,258929.20,2026-11-01 03:52:57.000549316
1,public__hotels.csv,.csv,211123.80,2026-11-01 03:53:21.363658667
2,public__sessions.csv,.csv,634810.70,2026-11-01 03:54:50.630051374
3,public__users.csv,.csv,67760.92,2026-11-01 03:54:58.960024118


### 4.4 Compare Database and Local Tables

Compare the tables available in the Neon database with the existing local original copies.

Classify each table according to its availability:

- Existing locally and in Neon: Skip download.
- Available in Neon but missing locally: Prepare for download.
- Available locally but missing in Neon: Retain local copy.
- Missing both in Neon and locally: No action required.

Existing local original copies must never be overwritten automatically.

In [94]:
# 4.4 Compare Database and Local Tables

print("TRAVELTIDE – DATABASE AND LOCAL DATA COMPARISON")
print("=" * 55)

# Create a set of local table names
local_table_names = {
    file.stem.replace("__", ".", 1)
    for file in original_files
}

# Check whether database information is available
if not database_info_available:
    print("Comparison: SKIPPED")
    print("Reason: Database table information is unavailable.")

else:
    # Create a set of database table names
    database_table_names = {
        f"{schema}.{table}"
        for schema, table in database_tables
    }

    # Compare database and local tables
    existing_locally = database_table_names & local_table_names
    missing_locally = database_table_names - local_table_names
    missing_in_database = local_table_names - database_table_names

    # Display comparison results
    print(f"Tables available in Neon: {len(database_table_names)}")
    print(f"Local original copies: {len(local_table_names)}")
    print(f"Existing locally: {len(existing_locally)}")
    print(f"Missing locally: {len(missing_locally)}")
    print(f"Missing in Neon: {len(missing_in_database)}")

    print("\nComparison completed.")

print("=" * 55)

TRAVELTIDE – DATABASE AND LOCAL DATA COMPARISON
Tables available in Neon: 4
Local original copies: 4
Existing locally: 4
Missing locally: 0
Missing in Neon: 0

Comparison completed.


### 4.5 Display Database and Local Data Status

Display a consolidated overview of the current database and local data status.

The status report includes:

- Database connection status.
- Number of tables available in Neon.
- Number of existing local original copies.
- Number of tables requiring a local download.
- Number of existing local copies that will be skipped.
- Number of local copies no longer available in Neon.

The report must clearly identify incomplete database configurations and unavailable database information.

In [95]:
# 4.5 Display Database and Local Data Status

print("TRAVELTIDE – DATABASE AND LOCAL DATA STATUS")
print("=" * 55)

# Display database configuration status
configuration_complete = len(missing_variables) == 0

print(
    "Database configuration:",
    "COMPLETE" if configuration_complete else "INCOMPLETE"
)

# Display actual database connection status
print(
    "Database connection:",
    "CONNECTED" if database_connected else "NOT CONNECTED"
)

# Display database and local data information
if database_info_available:

    print(f"Tables available in Neon: {len(database_tables)}")
    print(f"Local original copies: {len(local_table_names)}")
    print(f"Existing locally: {len(existing_locally)}")
    print(f"Tables requiring download: {len(missing_locally)}")
    print(f"Local copies missing in Neon: {len(missing_in_database)}")

else:

    print("Database table information: NOT AVAILABLE")
    print(f"Local original copies: {len(original_files)}")
    print("Tables requiring download: UNKNOWN")
    print("Existing copies to skip: UNKNOWN")
    print("Local copies missing in Neon: UNKNOWN")

print("=" * 55)

TRAVELTIDE – DATABASE AND LOCAL DATA STATUS
Database configuration: COMPLETE
Database connection: CONNECTED
Tables available in Neon: 4
Local original copies: 4
Existing locally: 4
Tables requiring download: 0
Local copies missing in Neon: 0


### 4.6 Import Missing Local Copies

Import database tables that are available in Neon but do not yet have a local original copy.

The import process follows these rules:

- Existing local original copies must be skipped.
- Missing local copies must be downloaded from Neon.
- Existing local files must never be overwritten automatically.
- Database tables without an existing local copy must be imported completely.
- Failed imports must be reported.
- Local original copies must remain unchanged during the import process.

All imported files are stored in the local original data directory.

In [96]:
# 4.6 Import Missing Local Copies

from psycopg import sql

print("TRAVELTIDE – LOCAL DATA IMPORT")
print("=" * 55)

# Initialize import results
imported_tables = []
skipped_tables = []
failed_tables = []

if not database_connected:
    print("Import: SKIPPED")
    print("Reason: No active database connection.")

elif not database_info_available:
    print("Import: SKIPPED")
    print("Reason: Database table information is unavailable.")

else:
    ORIGINAL_DIR.mkdir(parents=True, exist_ok=True)

    try:
        with psycopg.connect(**connection_parameters) as connection:

            # Enforce read-only database access
            with connection.cursor() as cursor:
                cursor.execute("SET TRANSACTION READ ONLY")

                for schema, table in database_tables:

                    # Create a unique filename for each database table
                    file_name = f"{schema}__{table}.csv"
                    target_file = ORIGINAL_DIR / file_name

                    # Skip existing local copies
                    if target_file.exists():
                        skipped_tables.append(file_name)
                        print(f"SKIPPED: {file_name}")
                        continue

                    # Track whether this import created the file
                    created_by_this_run = False

                    try:
                        # Create file exclusively to prevent overwriting
                        with target_file.open("xb") as output_file:
                            created_by_this_run = True

                            # Export table data as CSV
                            copy_query = sql.SQL(
                                "COPY {}.{} TO STDOUT "
                                "WITH (FORMAT CSV, HEADER TRUE, ENCODING 'UTF8')"
                            ).format(
                                sql.Identifier(schema),
                                sql.Identifier(table)
                            )

                            with cursor.copy(copy_query) as copy:
                                for data in copy:
                                    output_file.write(data)

                        imported_tables.append(file_name)
                        print(f"IMPORTED: {file_name}")

                    except FileExistsError:
                        # Another process created the file in the meantime
                        skipped_tables.append(file_name)
                        print(f"SKIPPED: {file_name}")

                    except Exception as error:
                        # Remove only an incomplete file created by this import
                        if created_by_this_run and target_file.exists():
                            target_file.unlink()

                        failed_tables.append(file_name)
                        print(f"FAILED: {file_name} – {error}")

    except psycopg.Error as error:
        print("Database import failed:", error)

    print("\nIMPORT RESULTS")
    print("-" * 55)
    print(f"New local copies: {len(imported_tables)}")
    print(f"Existing copies skipped: {len(skipped_tables)}")
    print(f"Failed imports: {len(failed_tables)}")

print("=" * 55)

TRAVELTIDE – LOCAL DATA IMPORT
SKIPPED: public__flights.csv
SKIPPED: public__hotels.csv
SKIPPED: public__sessions.csv
SKIPPED: public__users.csv

IMPORT RESULTS
-------------------------------------------------------
New local copies: 0
Existing copies skipped: 4
Failed imports: 0


### 4.7 Display Import Summary

Display a summary of the database import process.

The summary includes:

- Number of newly imported local copies.
- Number of existing local copies skipped.
- Number of failed imports.
- Overall import status.

If the database connection or import process was skipped, the summary must clearly indicate that no import was performed.

No existing local original copies may be overwritten or deleted.

In [97]:
# 4.7 Display Import Summary

print("TRAVELTIDE – IMPORT SUMMARY")
print("=" * 55)

# Check whether the import process was executed
if not database_connected:
    print("Import status: NOT PERFORMED")
    print("Reason: No active database connection.")

elif not database_info_available:
    print("Import status: NOT PERFORMED")
    print("Reason: Database table information is unavailable.")

else:
    # Retrieve import results
    imported_count = len(imported_tables)
    skipped_count = len(skipped_tables)
    failed_count = len(failed_tables)

    print(f"New local copies: {imported_count}")
    print(f"Existing copies skipped: {skipped_count}")
    print(f"Failed imports: {failed_count}")

    # Determine overall import status
    if failed_count > 0:
        print("Import status: COMPLETED WITH ERRORS")

    elif imported_count == 0 and skipped_count == 0:
        print("Import status: COMPLETED – NO TABLES FOUND")

    else:
        print("Import status: COMPLETED")

print("=" * 55)

TRAVELTIDE – IMPORT SUMMARY
New local copies: 0
Existing copies skipped: 4
Failed imports: 0
Import status: COMPLETED
